In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

In [1]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import cv2
from glob import glob
from tqdm import tqdm

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader

from torchvision import transforms, models
from torchvision.datasets import ImageFolder

from sklearn.metrics import confusion_matrix, classification_report

In [2]:
DATASET_PATH = "/kaggle/input/datasets/rtlmhjbn/ip02-dataset/classification"

TRAIN_DIR = os.path.join(DATASET_PATH, "train")
VAL_DIR   = os.path.join(DATASET_PATH, "val")
TEST_DIR  = os.path.join(DATASET_PATH, "test")

print("Train Classes:", os.listdir(TRAIN_DIR))

Train Classes: ['7', '47', '17', '81', '19', '22', '2', '35', '92', '50', '23', '87', '10', '5', '61', '36', '20', '45', '60', '27', '64', '41', '89', '39', '32', '98', '25', '42', '52', '75', '8', '38', '12', '94', '55', '49', '0', '31', '62', '53', '101', '70', '34', '18', '79', '85', '88', '65', '67', '78', '28', '66', '56', '72', '16', '13', '99', '26', '74', '15', '3', '90', '69', '77', '86', '95', '43', '91', '71', '1', '58', '59', '97', '30', '14', '76', '84', '4', '83', '82', '57', '9', '96', '46', '21', '44', '40', '80', '6', '11', '68', '63', '37', '51', '33', '100', '54', '48', '29', '24', '73', '93']


In [3]:
def apply_clahe(image):
    lab = cv2.cvtColor(image, cv2.COLOR_RGB2LAB)
    l, a, b = cv2.split(lab)

    clahe = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8,8))
    l = clahe.apply(l)

    merged = cv2.merge((l,a,b))
    return cv2.cvtColor(merged, cv2.COLOR_LAB2RGB)

In [4]:
class PestDataset(torch.utils.data.Dataset):
    def __init__(self, root_dir, transform=None):
        self.dataset = ImageFolder(root_dir)
        self.transform = transform

    def __len__(self):
        return len(self.dataset)

    def __getitem__(self, idx):
        img, label = self.dataset[idx]
        img = np.array(img)

        img = apply_clahe(img)

        if self.transform:
            img = self.transform(img)

        return img, label

In [5]:
train_transform = transforms.Compose([
    transforms.ToPILImage(),
    transforms.Resize((224,224)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(20),
    transforms.ColorJitter(),
    transforms.ToTensor()
])

val_transform = transforms.Compose([
    transforms.ToPILImage(),
    transforms.Resize((224,224)),
    transforms.ToTensor()
])

In [6]:
train_dataset = PestDataset(TRAIN_DIR, train_transform)
val_dataset   = PestDataset(VAL_DIR, val_transform)
test_dataset  = PestDataset(TEST_DIR, val_transform)

train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True, num_workers=2)
val_loader   = DataLoader(val_dataset, batch_size=32, shuffle=False, num_workers=2)
test_loader  = DataLoader(test_dataset, batch_size=32, shuffle=False, num_workers=2)

NUM_CLASSES = len(train_dataset.dataset.classes)
print("Total Classes:", NUM_CLASSES)

Total Classes: 102


In [7]:
class AdaptiveFusion(nn.Module):
    def __init__(self, dim):
        super().__init__()
        self.temperature = 2.0
        self.fc1 = nn.Linear(dim*2, dim)
        self.fc2 = nn.Linear(dim, 2)

    def forward(self, cnn_feat, vit_feat):
        combined = torch.cat([cnn_feat, vit_feat], dim=1)
        x = torch.relu(self.fc1(combined))
        logits = self.fc2(x)

        weights = torch.softmax(logits / self.temperature, dim=1)

        fused = weights[:,0:1]*cnn_feat + weights[:,1:2]*vit_feat
        return fused

In [8]:
class HybridModel(nn.Module):
    def __init__(self, num_classes):
        super().__init__()
        
        self.cnn = models.resnet50(pretrained=True)
        self.cnn.fc = nn.Identity()
        
        self.vit = models.vit_b_16(pretrained=True)
        self.vit.heads = nn.Identity()
        
        self.cnn_proj = nn.Linear(2048, 768)
        
        self.fusion = AdaptiveFusion(768)
        self.classifier = nn.Linear(768, num_classes)

    def forward(self, x):
        cnn_feat = self.cnn(x)
        vit_feat = self.vit(x)

        cnn_feat = self.cnn_proj(cnn_feat)

        fused = self.fusion(cnn_feat, vit_feat)
        return self.classifier(fused)

model = HybridModel(NUM_CLASSES).cuda()

/usr/local/lib/python3.12/dist-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=ResNet50_Weights.IMAGENET1K_V1`. You can also use `weights=ResNet50_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)


Downloading: "https://download.pytorch.org/models/resnet50-0676ba61.pth" to /root/.cache/torch/hub/checkpoints/resnet50-0676ba61.pth


100%|██████████| 97.8M/97.8M [00:00<00:00, 221MB/s]
/usr/local/lib/python3.12/dist-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=ViT_B_16_Weights.IMAGENET1K_V1`. You can also use `weights=ViT_B_16_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)


Downloading: "https://download.pytorch.org/models/vit_b_16-c867db91.pth" to /root/.cache/torch/hub/checkpoints/vit_b_16-c867db91.pth


100%|██████████| 330M/330M [00:01<00:00, 205MB/s] 


In [ ]:
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=3e-4)

train_losses, val_losses = [], []
train_accs, val_accs = [], []

for epoch in range(15):

    model.train()
    running_loss, correct, total = 0, 0, 0

    for imgs, labels in tqdm(train_loader):
        imgs, labels = imgs.cuda(), labels.cuda()

        optimizer.zero_grad()
        outputs = model(imgs)

        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        running_loss += loss.item()

        preds = outputs.argmax(1)
        correct += (preds == labels).sum().item()
        total += labels.size(0)

    train_loss = running_loss / len(train_loader)
    train_acc = correct / total

    train_losses.append(train_loss)
    train_accs.append(train_acc)

    # VALIDATION
    model.eval()
    val_running_loss, val_correct, val_total = 0, 0, 0

    with torch.no_grad():
        for imgs, labels in val_loader:
            imgs, labels = imgs.cuda(), labels.cuda()

            outputs = model(imgs)
            loss = criterion(outputs, labels)

            val_running_loss += loss.item()

            preds = outputs.argmax(1)
            val_correct += (preds == labels).sum().item()
            val_total += labels.size(0)

    val_loss = val_running_loss / len(val_loader)
    val_acc = val_correct / val_total

    val_losses.append(val_loss)
    val_accs.append(val_acc)

    print(f"\nEpoch {epoch+1}")
    print(f"Train Loss: {train_loss:.4f} | Train Acc: {train_acc:.4f}")
    print(f"Val Loss:   {val_loss:.4f} | Val Acc:   {val_acc:.4f}")

100%|██████████| 1410/1410 [37:47<00:00,  1.61s/it]



Epoch 1
Train Loss: 2.6149 | Train Acc: 0.3614
Val Loss:   2.2148 | Val Acc:   0.4358


100%|██████████| 1410/1410 [38:00<00:00,  1.62s/it]



Epoch 2
Train Loss: 1.9957 | Train Acc: 0.4809
Val Loss:   1.8572 | Val Acc:   0.5044


100%|██████████| 1410/1410 [37:57<00:00,  1.62s/it]


...

100%|██████████| 1410/1410 [38:00<00:00,  1.62s/it]



Epoch 13
Train Loss: 0.7887 | Train Acc: 0.7598
Val Loss:   1.5518 | Val Acc:   0.6308


100%|██████████| 1410/1410 [37:59<00:00,  1.62s/it]



Epoch 14
Train Loss: 0.7430 | Train Acc: 0.7719
Val Loss:   1.5444 | Val Acc:   0.6365


  0%|          | 3/1410 [00:05<41:46,  1.78s/it]